# About this notebook
> This is an exmaple implementation for finding the best single binding sites, using mCherry as target RNA in a Yeast cell. <br>
>  Special: transcriptome with expression levels and target RNA (mCherry) with ViennaRNA binding site availability scores

# Imports

In [1]:
import pandas as pd
from main_search import *
from sequence_reader import read_fasta, _ensembl_fasta_to_dict


# Lets evaluate the affinities for each possible binding site sequence of target against a transcriptome
 - evaluate all target-sequence substrings = queries of specific length against a reference dataset


#### General Fasta importer - mCherry Sequencing results

In [2]:
mCherry = read_fasta("sequence_data/iGEM_mCherry_V2.fasta").iloc[0]
mCherry.SEQUENCE = mCherry.SEQUENCE.upper()
mCherry

SEQ_NAME                               iGEM_mCherry_BBa_E2060
SEQUENCE    ATGGCAACTAGCGGCATGGTTAGTAAAGGAGAAGAAAATAACATGG...
Name: 0, dtype: object

#### read ensembl FASTA and decode description
 - for FASTA data from https://ftp.ensembl.org/pub/

In [3]:
files = ["sequence_data/Saccharomyces_cerevisiae.R64-1-1.ncrna.fa", "sequence_data/Saccharomyces_cerevisiae.R64-1-1.cdna.all.fa", "sequence_data/Saccharomyces_cerevisiae.R64-1-1.cds.all.fa"]  # -> different transcripts in each DataFrame -> data has to be concatenated
_df = pd.concat([read_fasta(file) for file in files], ignore_index=True)
df_seq = pd.concat([_df.SEQ_NAME.apply(_ensembl_fasta_to_dict).apply(pd.Series), _df.SEQUENCE], axis = 1).drop_duplicates(ignore_index = True) # split description into respective columns i.e. decode ensebl fasta description
df_seq

,SEQ_NAME,chromosome,gene,gene_biotype,transcript_biotype,SEQUENCE
0,ETS2-2_rRNA,R64-1-1:XII:460712:460922:-1,ETS2-2,rRNA,NaN,TTTTTATTTCTTTCTAAGTGGGTACTGGCAGGAGCCGGGGCCTAGT...
1,ITS2-2_rRNA,R64-1-1:XII:464319:464550:-1,ITS2-2,rRNA,NaN,CCTTCTCAAACATTCTGTTTGGTAGTGAGTGATACTCTTTGGAGTT...
2,RDN18-1_rRNA,R64-1-1:XII:455933:457732:-1,RDN18-1,rRNA,NaN,TATCTGGTTGATCCTGCCAGTAGTCATATGCTTGTCTCAAAGATTA...
3,Q0020_rRNA,R64-1-1:Mito:6546:8194:1,Q0020,rRNA,rRNA,GTAAAAAATTTATAAGAATATGATGTTGGTTCAGATTAAGCGCTAA...
4,RDN5-5_rRNA,R64-1-1:XII:485697:485815:1,RDN5-5,rRNA,NaN,GGTTGCGGCCATATCTACCAGAAAGCACCGTTTCCCGTCCGATCAA...
...,...,...,...,...,...,...
7035,YIL170W,R64-1-1:IX:19847:21220:1,YIL170W,pseudogene,pseudogene,ATGGGTTTGATTGTCTCAATATTCAACATTGGCTGCGCCATAGGCG...
7036,YMR242C_mRNA,R64-1-1:XIII:753225:753742:-1,YMR242C,protein_coding,protein_coding,GCTCACTTTAAAGAATACCAAGTTATTGGCCGTCGTTTGCCAACTG...
7037,YOR312C_mRNA,R64-1-1:XV:900250:900767:-1,YOR312C,protein_coding,protein_coding,GCTCATTTCAAAGAATACCAAGTCATTGGTCGTCGTTTACCAACTG...
7038,YJL041W_mRNA,R64-1-1:X:365903:368373:1,YJL041W,protein_coding,protein_coding,AACTTCAATACACCTCAACAAAACAAAACGCCCTTTTCGTTCGGGA...


#### read TPM.tsv files - quantitative transcriptome
data from https://www.ebi.ac.uk/gxa/experiments?species=%22saccharomyces+cerevisiae%22&experimentType=%22Baseline%22

In [4]:
files = ["sequence_data/E-MTAB-8621-query-results.tpms.tsv", "sequence_data/E-MTAB-8626-query-results.tpms.tsv"] # same genes under different conditions -> data has to be merged
_df = pd.read_csv(files[0], sep = '\t', skiprows = 4)
for file in files[1:]:
    _df = pd.merge(_df, pd.read_csv(file, sep = '\t', skiprows = 4) , how = "outer", on = ["Gene ID", "Gene Name"])

df_tpm = _df.drop(columns = _df.filter(like="minute").columns) # combine timeseries data by taking the max of the found expression
df_tpm["tpm_max"] = _df.filter(like="minute").max(axis = 1, skipna = True, numeric_only = True)
df_tpm

,Gene ID,Gene Name,tpm_max
0,ETS1-1,ETS1-1,72.0
1,ETS1-2,ETS1-2,3.0
2,ETS2-1,ETS2-1,8.0
3,HRA1,HRA1,3.0
4,ICR1,ICR1,5.0
...,...,...,...
6782,tY(GUA)F2,SUP6,3.0
6783,tY(GUA)J1,SUP7,9.0
6784,tY(GUA)J2,SUP4,3.0
6785,tY(GUA)M1,SUP5,6.0


### merge sequence and tpm data

In [5]:
transcriptome = pd.merge(df_seq, df_tpm, left_on="gene", right_on="Gene ID", how = "outer") 
transcriptome = transcriptome[~transcriptome.SEQUENCE.isna()]                               # delete entries witho9ut a sequence
transcriptome.loc[transcriptome.tpm_max.isna(), "tpm_max"] = transcriptome.tpm_max.max()                   # if transcript count (TPM) is unknown, expect the worst
transcriptome["WEIGHT"] = transcriptome.tpm_max
mCherry["WEIGHT"] = transcriptome.tpm_max.mean()
transcriptome

,SEQ_NAME,chromosome,gene,gene_biotype,transcript_biotype,SEQUENCE,Gene ID,Gene Name,tpm_max,WEIGHT
0,ETS1-1_rRNA,R64-1-1:XII:457733:458432:-1,ETS1-1,rRNA,NaN,ATGCGAAAGCAGTTGAAGACAAGTTCGAAAAGAGTTTGGAAACGAA...,ETS1-1,ETS1-1,72.0,72.0
1,ETS1-2_rRNA,R64-1-1:XII:466870:467569:-1,ETS1-2,rRNA,NaN,ATGCGAAAGCAGTTGAAGACAAGTTCGAAAAGAGTTTGGAAACGAA...,ETS1-2,ETS1-2,3.0,3.0
2,ETS2-1_rRNA,R64-1-1:XII:451575:451785:-1,ETS2-1,rRNA,NaN,TTTTTATTTCTTTCTAAGTGGGTACTGGCAGGAGCCGGGGCCTAGT...,ETS2-1,ETS2-1,8.0,8.0
3,ETS2-2_rRNA,R64-1-1:XII:460712:460922:-1,ETS2-2,rRNA,NaN,TTTTTATTTCTTTCTAAGTGGGTACTGGCAGGAGCCGGGGCCTAGT...,NaN,NaN,22546.0,22546.0
4,HRA1_ncRNA,R64-1-1:I:99305:99868:1,HRA1,ncRNA,NaN,GGGCCCTTTCTTCCGTTTGAACGTAAAGGCATTTTTGAGACCATTA...,HRA1,HRA1,3.0,3.0
...,...,...,...,...,...,...,...,...,...,...
7083,tY(GUA)J2_tRNA,R64-1-1:X:542956:543044:-1,tY(GUA)J2,tRNA,tRNA,CTCTCGGTAGCCAAGTTGGTTTAAGGCGCAAGACTGTAAATCTTGA...,tY(GUA)J2,SUP4,3.0,3.0
7084,tY(GUA)M1_tRNA,R64-1-1:XIII:168795:168883:1,tY(GUA)M1,tRNA,tRNA,CTCTCGGTAGCCAAGTTGGTTTAAGGCGCAAGACTGTAAATCTTGA...,tY(GUA)M1,SUP5,6.0,6.0
7085,tY(GUA)M2_tRNA,R64-1-1:XIII:837928:838016:1,tY(GUA)M2,tRNA,tRNA,CTCTCGGTAGCCAAGTTGGTTTAAGGCGCAAGACTGTAAATCTTGA...,NaN,NaN,22546.0,22546.0
7086,tY(GUA)O_tRNA,R64-1-1:XV:288192:288280:1,tY(GUA)O,tRNA,tRNA,CTCTCGGTAGCCAAGTTGGTTTAAGGCGCAAGACTGTAAATCTTGA...,tY(GUA)O,SUP3,3.0,3.0


### Load mcherry secondary structure scores

In [6]:
def iseq(seq):
    "invert a sequence: bases and direction"
    return seq.lower().replace("a", "T").replace("t", "A").replace("c", "G").replace("g", "C")[::-1]

In [7]:
mCherry_secstructure = pd.read_csv("sequence_data/mcherry-rna-secstructure-CasRxguides.csv").rename(columns={'GuideSeq': 'SEQUENCE', 'GuideName': 'SEQ_NAME', 'MatchPos':'POSITION' })
mCherry_secstructure["WEIGHT"] = 1
mCherry_secstructure.SEQUENCE = mCherry_secstructure.SEQUENCE.apply(iseq)
mCherry_secstructure.loc[:,["GuideScores", "Rank", "standardizedGuideScores"]] *= -1
mCherry_secstructure

,SEQ_NAME,SEQUENCE,POSITION,GuideScores,Rank,standardizedGuideScores,quartiles,WEIGHT
0,crRNA336:336-358,TGGCGTGGTAACAGTGACTCAGG,358,-1.358254,-0.998600,-0.913186,4,1
1,crRNA515:536-558,ATGGTGGACATTATGACGCTGAG,558,-1.323466,-0.997200,-0.893866,4,1
2,crRNA580:601-623,GGTGCTTACAATGTAAATATAAA,623,-1.315502,-0.995900,-0.889443,4,1
3,crRNA578:599-621,CAGGTGCTTACAATGTAAATATA,621,-1.313184,-0.994500,-0.888156,4,1
4,crRNA516:537-559,TGGTGGACATTATGACGCTGAGG,559,-1.300853,-0.993100,-0.881308,4,1
...,...,...,...,...,...,...,...,...
719,crRNA167:167-189,TAAAAGTTACTAAGGGTGGCCCA,189,0.129648,-0.005525,-0.086875,1,1
720,crRNA382:382-404,TTTATCTACAAAGTCAAATTAAG,404,0.129760,-0.004144,-0.086812,1,1
721,crRNA166:166-188,CTAAAAGTTACTAAGGGTGGCCC,188,0.132534,-0.002762,-0.085272,1,1
722,crRNA381:381-403,ATTTATCTACAAAGTCAAATTAA,403,0.136348,-0.001381,-0.083154,1,1


<br><br><br>


# Lets evaluate the affinities for each possible binding site sequence of target against a transcriptome
 - evaluate all target-sequence substrings = queries of specific length against a reference dataset


In [8]:
# generate random input sequences for this test
reference_dataset = transcriptome        # The references to check against, i.e. the transcriptome in this example
target            = mCherry              # The sequence from which we want ot find the best/unique binding site
query_len = 20                           # length of the binding sites (usually 8-12 for pumby)

# only check possible binding sites with good secondary structure accessibility
query_dataset = mCherry_secstructure[mCherry_secstructure.quartiles == 4]


# compare possible binding sites against whole transcriptome
query_sc_summary(query_dataset.sample().iloc[0], reference_dataset, target) # testrun to make sure everything is ok, because for some stupid reason parallelised executions do not return warnings anymore
DF = pd.DataFrame(list(tqdm(total=len(query_dataset), ncols = 100, 
        iterable=Parallel(return_as="generator", n_jobs=-1)(  # using joblib instead of multiprocessing becasue of windows compatibility
            delayed(query_sc_summary)(query, reference_dataset, target) for _, query in query_dataset.iterrows())
        )))

100%|█████████████████████████████████████████████████████████████| 138/138 [00:23<00:00,  5.84it/s]


# Visualize results 
- thats a lot of results, lets try to understand what we got
- remember the goal: find the queries with the lowest chance of binding in the trancsriptome

In [9]:
df_genRanks(DF)
df_select_against_outliers(DF, 8); # select 16 best

In [10]:
#df_PCA(DF)
df_table(DF) 

/mnt/Data/GIT/igem_software/main_search.py:431: RuntimeWarning: invalid value encountered in log10
  mag = -int(np.floor(np.log10(x))-digits+1)


**table explanation**
 - first column is the position on the target
 - in the other columns red entries are worse than average, grey is average, blue (lower affinity to transcriptome) is better
 - columns starting with 'W.' used the WEIGHT column (e.g. expression level) provided to weight the contribution of each reference i.e. transcript on the overall result
 - each metric has max and sum result: sum is the sum over all references i.e. e.g. the affinity (boltzmann factor) of the whole transcriptome, while max is the worst affinity of a single transcript
 - Boltzmann factors can be understood as binding affinities. When column name is followed by 'r2t' then these results are relative affinities compared to target, i.e. an max affinity of 0.1 means it's 10x less likely to bind to the wordt reference, but a sum of 6 means its 6x more likely to bind to the transcriptome than to the target
 - rank is calculated based on all other columns.
 - SPECIAL HERE: the first guide scores are from vienna RNA. Because the table is formatted to give blue (good) to  small values, I had to flip the sign on all Vienna Scores